In [ ]:
#!git clone -b update-gluonts https://github.com/time-series-foundation-models/lag-llama/

In [ ]:
cd lag-llama

In [ ]:
#!pip install -r requirements.txt  # this could take some time # ignore the errors displayed by colab

In [ ]:
#!pip install -U torch torchvision

In [ ]:
#!huggingface-cli download time-series-foundation-models/Lag-Llama lag-llama.ckpt --local-dir lag-llama

In [ ]:
from itertools import islice

from matplotlib import pyplot as plt
import matplotlib.dates as mdates

import torch
from gluonts.evaluation import make_evaluation_predictions, Evaluator
from gluonts.dataset.repository.datasets import get_dataset

from gluonts.dataset.pandas import PandasDataset
import pandas as pd
import numpy as np
from gluonts.dataset.common import ListDataset

from lag_llama.gluon.estimator import LagLlamaEstimator

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*non-tuple sequence for multidimensional indexing.*")

In [ ]:
import sys
from types import ModuleType

# Create dummy module hierarchy
def create_dummy_module(module_path):
    """
    Create a dummy module hierarchy for the given path.
    Returns the leaf module.
    """
    parts = module_path.split('.')
    current = ''
    parent = None

    for part in parts:
        current = current + '.' + part if current else part
        if current not in sys.modules:
            module = ModuleType(current)
            sys.modules[current] = module
            if parent:
                setattr(sys.modules[parent], part, module)
        parent = current

    return sys.modules[module_path]

# Create the dummy gluonts module hierarchy
gluonts_module = create_dummy_module('gluonts.torch.modules.loss')

# Create dummy classes for the specific loss functions
class DistributionLoss:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        return 0.0

    def __getattr__(self, name):
        return lambda *args, **kwargs: None

class NegativeLogLikelihood:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        return 0.0

    def __getattr__(self, name):
        return lambda *args, **kwargs: None

# Add the specific classes to the module
gluonts_module.DistributionLoss = DistributionLoss
gluonts_module.NegativeLogLikelihood = NegativeLogLikelihood

In [ ]:
def get_lag_llama_predictions(dataset, prediction_length, device, context_length=32, use_rope_scaling=False, num_samples=100):
    ckpt = torch.load("lag-llama.ckpt", map_location=device, weights_only=False) # Uses GPU since in this Colab we use a GPU.
    estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

    rope_scaling_arguments = {
        "type": "linear",
        "factor": max(1.0, (context_length + prediction_length) / estimator_args["context_length"]),
    }

    estimator = LagLlamaEstimator(
        ckpt_path="lag-llama.ckpt",
        prediction_length=prediction_length,
        context_length=context_length, # Lag-Llama was trained with a context length of 32, but can work with any context length

        # estimator args
        input_size=estimator_args["input_size"],
        n_layer=estimator_args["n_layer"],
        n_embd_per_head=estimator_args["n_embd_per_head"],
        n_head=estimator_args["n_head"],
        scaling=estimator_args["scaling"],
        time_feat=estimator_args["time_feat"],
        rope_scaling=rope_scaling_arguments if use_rope_scaling else None,

        batch_size=1,
        num_parallel_samples=100,
        device=device,
    )

    lightning_module = estimator.create_lightning_module()
    transformation = estimator.create_transformation()
    predictor = estimator.create_predictor(transformation, lightning_module)

    forecast_it, ts_it = make_evaluation_predictions(
        dataset=dataset,
        predictor=predictor,
        num_samples=num_samples
    )
    forecasts = list(forecast_it)
    tss = list(ts_it)

    return forecasts, tss

In [ ]:
def build_lag_llama_predictor(prediction_length, device, context_length=32, use_rope_scaling=False, num_samples=100):
    ckpt = torch.load("lag-llama.ckpt", map_location=device, weights_only=False)
    estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

    rope_scaling_arguments = {
        "type": "linear",
        "factor": max(1.0, (context_length + prediction_length) / estimator_args["context_length"]),
    }


    estimator = LagLlamaEstimator(
        ckpt_path="lag-llama.ckpt",
        prediction_length=prediction_length,
        context_length=context_length,
        input_size=estimator_args["input_size"],
        n_layer=estimator_args["n_layer"],
        n_embd_per_head=estimator_args["n_embd_per_head"],
        n_head=estimator_args["n_head"],
        scaling=estimator_args["scaling"],
        time_feat=estimator_args["time_feat"],
        nonnegative_pred_samples=True,
        rope_scaling=rope_scaling_arguments if use_rope_scaling else None,
        batch_size=1,
        num_parallel_samples=1,  
    )

    lightning_module = estimator.create_lightning_module()
    transformation = estimator.create_transformation()
    predictor = estimator.create_predictor(transformation, lightning_module)

    return predictor


In [ ]:
from pathlib import Path

dataset_name = "webbrowsing_train.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"], index_col="DATE")
print("Loaded:", data_file)

In [ ]:
# Set numerical columns as float32
for col in df.columns:
    # Check if column is not of string type
    if df[col].dtype != 'object' and pd.api.types.is_string_dtype(df[col]) == False:
        df[col] = df[col].astype('float32')

train_end = round(len(df) * 0.8)

train = PandasDataset(df[:train_end], freq="ms", target="mac_dl_brate")
test = PandasDataset(df[train_end:], freq="ms", target="mac_dl_brate")

In [ ]:
prediction_length=96
context_length = 5
num_samples = 1
device = torch.device("cpu")
batch_size = 128

In [ ]:
def rolling_evaluation(dataset, predictor, prediction_length=96, stride=1):
    ts_entry = next(iter(dataset))  
    values = ts_entry["target"]

    start_time = (
        ts_entry["start"].to_timestamp()
        if isinstance(ts_entry["start"], pd.Period)
        else pd.Timestamp(ts_entry["start"])
    )

    index = pd.date_range(start=start_time, periods=len(values), freq=dataset.freq)

    contexts = []
    for start in range(0, len(values) - prediction_length, stride):
        context = values[: start + prediction_length]
        contexts.append({"target": context, "start": start_time})

    tmp_dataset = ListDataset(contexts, freq=dataset.freq)
    forecasts = list(predictor.predict(tmp_dataset))

    results = []
    for i, forecast in enumerate(forecasts):
        forecast_mean = forecast.mean_ts.values
        context_end = i + prediction_length

        max_len = min(prediction_length, len(values) - context_end)

        forecast_ts = index[context_end : context_end + max_len]
        forecast_mean = forecast_mean[:max_len]
        actual = values[context_end : context_end + max_len]

        df_forecast = pd.DataFrame({
            "timestamp": forecast_ts,
            "mean": forecast_mean,
            "actual": actual,
        })
        results.append(df_forecast)

    return pd.concat(results, ignore_index=True)

In [ ]:
predictor = build_lag_llama_predictor(
    prediction_length=prediction_length,
    context_length=context_length,
    device=device
)

In [ ]:
forecasts_df = rolling_evaluation(
    test,
    predictor=predictor,
    prediction_length=prediction_length,
    stride=1
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

train_values = []
for ts_entry in train: 
    train_values.extend(ts_entry["target"])

train_values = np.array(train_values).reshape(-1, 1)

scaler_target = MinMaxScaler()
scaler_target.fit(train_values)

actual_scaled = scaler_target.transform(forecasts_df[["actual"]].values)
pred_scaled = scaler_target.transform(forecasts_df[["mean"]].values)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(actual_scaled, pred_scaled))
mae = mean_absolute_error(actual_scaled, pred_scaled)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

In [ ]:
df_avg = (
    forecasts_df
    .groupby("timestamp")
    .agg({"mean": "mean", "actual": "mean"})
    .reset_index()
)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df_avg["timestamp"], df_avg["actual"], label="Actual")
plt.plot(df_avg["timestamp"], df_avg["mean"], label="Predicted")
plt.legend()
plt.show()

In [ ]:
results_dir = Path("../results/metrics/web_browsing")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    "model": "lag-Llama_Zeroshot",
    "setting": "univariate",
    "dataset": "web_browsing",
    "rmse": rmse,
    "mae": mae,
}])

metrics_file = results_dir / "laglamazs_uni_metrics.csv"
metrics_df.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)